# **DLO-JZ Optimisation de l'apprentissage**
<img src="./images/optimisation.png" style="float: left; margin-right: 1em;"/>

<div class="alert alert-block alert-success">
    
## Objet des notebooks

Le but de ces trois *notebooks* est d'optimiser un code d'apprentissage d'un modèle *Resnet-50* sur *Imagenet* pour Jean Zay en implémentant :
* **TP2.1** : l'optimisation du *Dataloader*
* **TP2.2** : la DDP (*Distributed Data Parallelism*)
* **TP2.3** : le DDP et le problème des paramètres non utilisés


Les cellules dans ce *notebook* ne sont pas prévues pour être modifiées, sauf rares exceptions indiquées dans les commentaires. Les TP se feront en modifiant les codes `dlojz1_X.py`.

Les directives de modification seront marquées par l'étiquette suivante <div class="alert alert-block alert-warning">**TODO**</div>
Des solutions sont présentes dans le répertoire `solutions/`.

*Notebook rédigé par l'équipe assistance IA de l'IDRIS, mai 2026.*

</div>

### **Environnement de calcul**

Les fonctions *python* de gestion de queue SLURM développées par l'IDRIS et les fonctions dédiées à la formation DLO-JZ sont à importer.

Le module d'environnement pour les *jobs* et la taille des images sont fixés pour ce *notebook*.
<div class="alert alert-block alert-warning">
    
**TODO :** choisir un pseudonyme (maximum 5 caractères) pour vous différencier dans la queue SLURM pendant la formation.

</div>

In [ ]:
from idr_pytools import display_slurm_queue, gpu_jobs_submitter, search_log
from dlojz_tools import controle_technique, compare, comm_profiler, turbo_profiler, BatchNorm_view, metric_compute_log
MODULE = 'pytorch-gpu/py3/2.8.0'
image_size = 224
account = 'for@a100'
name = 'pseudo'   ## Pseudonyme à choisir
!mkdir -p checkpoints # Création d'un répertoire `checkpoints/` si cela n'a pas déjà été fait.

### **Gestion de la queue SLURM**

Pour afficher vos jobs dans la queue SLURM :

In [ ]:
display_slurm_queue(name)

**Remarque**: cette fonction sera utilisée plusieurs fois dans ce *notebook*. Elle permet d'afficher la queue de manière dynamique, rafraichie toutes les 5 secondes. Elle ne s'arrête que lorsque la queue est vide. Si vous désirez reprendre la main sur le *notebook*, il vous suffira d'arrêter manuellement la cellule avec le bouton *stop*. Cela n'a bien sûr aucun impact sur les *jobs* soumis.

Si vous voulez retirer TOUS vos *jobs* de la queue SLURM, décommenter et exécuter la cellule suivante :

In [ ]:
#!scancel -u $USER

Si vous voulez retirer UN de vos *jobs* de la queue SLURM, décommenter, compléter et exécuter la cellule suivante :

In [ ]:
#!scancel <jobid>

<div class="alert alert-block alert-success">


# TP2_3 : DDP Advanced Parameters

<div class="alert alert-block alert-warning">

**TODO**: Voir la [documentation Pytorch](https://docs.pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html#torch.nn.parallel.DistributedDataParallel) sur les paramètres de la classe `torch.nn.parallel.DistributedDataParallel`.

</div>

Les paramètres qui ne sont pas abordés dans la [documentation de l'IDRIS](http://www.idris.fr/docs/jean-zay/intelligence_artificielle/distribution_parallelisme/data-parallelism-pytorch), ont peu d'intérêts, pour les petits modèles de la taille d'un Resnet-152.

Cependant dans certains cas particuliers, le paramètre `find_unused_parameters` devient nécessaire. Nous allons donc étudier ce paramètre dans ce TP.

</div>

<div class="alert alert-block alert-info">

## Find unused Parameters
De la documentation Pytorch :
> **find_unused_parameters (bool)** – Traverse the autograd graph from all tensors contained in the return value of the wrapped module’s forward function. Parameters that don’t receive gradients as part of this graph are preemptively marked as being ready to be reduced. In addition, parameters that may have been used in the wrapped module’s forward function but were not part of loss computation and thus would also not receive gradients are preemptively marked as ready to be reduced. (default: False)

</div>

<div class="alert alert-block alert-info">

### 1. Stochastic Multi-head Classifier
<img src="./images/hydra.png" style="float: left; margin-right: 1em;" width=128 height=128/>  

Nous prenons ici, un modèle fantaisiste pour illustrer facilement un cas particulier de *unused parameters*.

A la fin d'un *Resnet-152* nous ajoutons plusieurs têtes de classifications. Lors de chaque *forward*, une seule tête sera choisi aléatoirement.

Nous sommes donc dans un cas où aléatoirement certaines couches, et donc certains **paramètres ne sont pas utilisés**. Observons ce qu'il se passe !

Dans le fichier [dlojz1_3_1.py](./dlojz1_3_1.py), on initialise le modèle de la manière suivante, en utilisant le fichier [custom_models.py](./custom_models.py):

```python
import torchvision.models as models 
from custom_models import add_conditional_heads

model = models.resnet152()
model = add_conditional_heads(model, n_heads=3)
```

</div>

## Garage - Mise à niveau
On fixe la taille d'image pour ce TP et le batch size optimal d'après les expériences du Jour 1

In [ ]:
image_size = 224
bs_optim = 512

### Soumission du job avec `find_unused_parameters=False`.
<u>**Attention vous sollicitez les noeuds de calcul à ce moment-là**.</u>

Pour soumettre le job, veuillez basculer la cellule suivante du mode `Raw NBConvert` au mode `Code` (raccourci: touche `Y` avec la cellule sélectionnée)

In [ ]:
command = f'./dlojz1_3_1.py -b {bs_optim} --image-size {image_size} --test'
n_gpu = 4
jobid = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                    account=account, time_max='00:10:00')
print(f'jobid = {jobid}')

Puis, rebasculer la cellule précédente en mode `Raw NBConvert`, afin d'eviter de relancer un job par erreur (raccourci: touche `R` avec la cellule sélectionnée).

In [ ]:
display_slurm_queue(name)

In [ ]:
controle_technique(jobid)

<div class="alert alert-block alert-warning">

**TODO** : Aller lire le *log d'erreur* pour interpréter l'erreur remontée.

</div>

### Soumission du job avec `find_unused_parameters=True`.
<u>**Attention vous sollicitez les noeuds de calcul à ce moment-là**.</u>

Pour soumettre le job, veuillez basculer la cellule suivante du mode `Raw NBConvert` au mode `Code` (raccourci: touche `Y` avec la cellule sélectionnée)

In [ ]:
command = f'./dlojz1_3_1.py -b {bs_optim} --image-size {image_size} --test --find-unused-parameters'
n_gpu = 4
jobid = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                    account=account, time_max='00:10:00')
print(f'jobid = {jobid}')

Puis, rebasculer la cellule précédente en mode `Raw NBConvert`, afin d'eviter de relancer un job par erreur (raccourci: touche `R` avec la cellule sélectionnée).

In [ ]:
display_slurm_queue(name)

In [ ]:
controle_technique(jobid)

<div class="alert alert-block alert-info">

### 2. Stochastic Depth
<img src="./images/Stochastic-Depth.png" style="float: left; margin-right: 1em;" width=640/>  

Le *Stochastic Depth* est une technique de *regularization* lors de l'apprentissage qui *drop* aléatoirement des *layers* entiers avec une probabilité de plus en plus forte selon la profondeur de la couche.

Nous sommes donc dans un nouveau cas de **paramètres non utilisés** aléatoirement, que nous allons testé de la même manière que précedemment.

Dans le fichier [dlojz1_3_2.py](./dlojz1_3_2.py), on initialise le modèle de la manière suivante, en utilisant le fichier [custom_models.py](./custom_models.py):

<br></br>
```python
from custom_models import resnet152_with_stochastic_depth

model = resnet152_with_stochastic_depth()
```

</div>

### Soumission du job avec `find_unused_parameters=False`.
<u>**Attention vous sollicitez les noeuds de calcul à ce moment-là**.</u>

Pour soumettre le job, veuillez basculer la cellule suivante du mode `Raw NBConvert` au mode `Code` (raccourci: touche `Y` avec la cellule sélectionnée)

In [ ]:
command = f'./dlojz1_3_2.py -b {bs_optim} --image-size {image_size} --test'
n_gpu = 4
jobid = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                    account=account, time_max='00:10:00')
print(f'jobid = {jobid}')

Puis, rebasculer la cellule précédente en mode `Raw NBConvert`, afin d'eviter de relancer un job par erreur (raccourci: touche `R` avec la cellule sélectionnée).

In [ ]:
display_slurm_queue(name)

In [ ]:
controle_technique(jobid)

<div class="alert alert-block alert-info">

**Remarque**: Ici, on utilise des *masks* pour *drop* certaines *layers*. Tous les gradients — ainsi que les activations — sont donc bien calculés, puis remis à zéro lorsqu’ils sont masqués. Le graphe de calcul reste complet et identique pour chaque passage, ce qui évite tout problème de synchronisation des gradients entre processus.
Dans ce contexte, le paramètre `find_unused_parameters` de **DDP** n’est pas nécessaire.

En revanche, pour des architectures réellement dynamiques, comme les modèles à **routage conditionnel** (*Mixture-of-Experts*, *Switch Transformers*) ou les **classifieurs multi-head** activant des branches différentes selon l’entrée, l’approche par masquage devient coûteuse et peu réaliste.
Dans ces cas, il est préférable d’activer `find_unused_parameters=True`, afin de permettre à DDP de gérer automatiquement les paramètres non utilisés lors du passage avant-arrière. 

</div>

<div class="alert alert-block alert-danger">

Arrêtez-vous ici! Une présentation vous attend avant le prochain TP.

</div>

<img src="./images/stop.png" style="float: left; margin-right: 1em;"/>